# Resultados y figuras para la presentación
## YOLO + SAHI sobre VisDrone — FLISOL 2026

Este notebook produce **todo el material cuantitativo** de la charla:

| Figura | Qué muestra | Slide |
|---|---|---|
| `00_pipeline_sahi.png` | Diagrama del pipeline de slicing | Teoría |
| `01_map_comparison.png` | mAP global YOLO vs YOLO+SAHI | Resultados |
| `02_ap_por_tamano.png` | AP por tamaño (small/medium/large) | Resultados ⭐ |
| `03_slice_tradeoff.png` | Detecciones vs tiempo según slice | Trade-offs |
| `04_postprocess.png` | NMS vs NMM vs GREEDYNMM | Postprocesamiento |
| `05_per_class.png` | Detecciones por clase (personas, motos...) | Caso de uso |
| `gallery/*.jpg` | Comparativas antes/después | Demo |
| `segmentation/*.jpg` | Segmentación antes/después | Segmentación |

> ⚠️ **REQUISITO**: el mAP real necesita el modelo entrenado en VisDrone.
> Si no lo tienes, corre primero **`00_entrenamiento.ipynb`**. Sin él, las figuras de mAP
> saldrán ~0 (zero-shot); las demás (pipeline, trade-off, galería, segmentación) funcionan igual.

> ⚙️ Activa la GPU (T4) antes de empezar.

---
## 1. Setup

In [ ]:
!pip install -q ultralytics sahi pycocotools matplotlib
!git clone https://github.com/TU-USUARIO/yolo-sahi-flisol2026.git || echo 'ya clonado'
%cd yolo-sahi-flisol2026
import os, sys; sys.path.insert(0, 'src')
import torch; print('GPU:', torch.cuda.is_available())

---
## 2. Datos + modelo

Descarga VisDrone val, lo convierte a COCO ground truth y recupera el modelo entrenado.
Si entrenaste con `00_entrenamiento.ipynb` y guardaste en Drive, se recupera automáticamente.

In [ ]:
# Recupera el modelo entrenado (00_entrenamiento.ipynb) desde Drive si está disponible
import shutil
try:
    from google.colab import drive; drive.mount('/content/drive')
    src = '/content/drive/MyDrive/flisol2026/visdrone_yolo11s.pt'
    if os.path.exists(src):
        os.makedirs('weights', exist_ok=True); shutil.copy(src, 'weights/visdrone_yolo11s.pt')
        print('Peso VisDrone recuperado de Drive')
except Exception as e:
    print('(sin Drive)', e)

!python scripts/download_visdrone.py
MODEL = 'weights/visdrone_yolo11s.pt' if os.path.exists('weights/visdrone_yolo11s.pt') else 'yolo11s.pt'
HAS_VISDRONE = os.path.exists('weights/visdrone_yolo11s.pt')
print('Modelo:', MODEL, '| mAP real disponible:', HAS_VISDRONE)
if not HAS_VISDRONE:
    print('AVISO: sin modelo VisDrone el mAP saldra ~0 (zero-shot). Entrena con 00_entrenamiento.ipynb.')

---
## 3. Benchmark principal — mAP YOLO vs YOLO + SAHI

Corre el mismo modelo en modo estándar y con SAHI sobre el val, y evalúa ambos con `pycocotools`.
La tabla resultante (`outputs/benchmark_map.csv`) es el **corazón numérico de la charla**.
Ajusta `--limit` según el tiempo (100 imágenes ≈ unos minutos en T4).

> Si ves la advertencia de desalineación de clases, es que el modelo NO está entrenado en VisDrone.

In [ ]:
!python src/evaluate.py --model $MODEL --limit 100
import pandas as pd
pd.read_csv('outputs/benchmark_map.csv') if os.path.exists('outputs/benchmark_map.csv') else 'sin resultados'

---
## 4. Trade-off del tamaño de slice

Cuántas detecciones y cuánto tiempo según el tamaño de slice. Funciona con cualquier modelo.

In [ ]:
!python src/slice_sweep.py --model $MODEL --limit 15 --sizes 320 512 640 768 1024

---
## 5. Comparación de postprocesamiento (NMS / NMM / GREEDYNMM)

Mismo slicing, distinto método de fusión de duplicados. El mAP requiere el modelo VisDrone;
el conteo de detecciones funciona siempre.

In [ ]:
!python src/compare_postprocess.py --model $MODEL --limit 60

---
## 6. Análisis por clase — ¿dónde ayuda más SAHI?

Para el caso de **seguridad ciudadana** importa saber el salto concreto en `pedestrian`, `people` y `motor`.
Este análisis cuenta detecciones por clase (YOLO vs SAHI).

In [ ]:
!python src/analyze_classes.py --model $MODEL --limit 60

---
## 7. Generar TODAS las figuras

In [ ]:
import importlib, figures; importlib.reload(figures)
figures.all_figures()

### 7.1 Previsualizar las figuras

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('outputs/figures/*.png')):
    print(f)
    display(Image(f))

---
## 8. Galería de detección antes/después

Comparativas lado a lado ordenadas por cuántos objetos extra encontró SAHI.

In [ ]:
!python src/gallery.py --model $MODEL --limit 25 --top 6
for f in sorted(glob.glob('outputs/gallery/*.jpg'))[:3]:
    print(f); display(Image(f))

---
## 9. Segmentación antes/después (cualitativo)

Genera paneles de **segmentación de instancias** con `yolo11s-seg.pt`: YOLO-seg vs YOLO-seg + SAHI.
Ideal para mostrar que SAHI también mejora las **máscaras** de objetos pequeños.

In [ ]:
!python src/segment.py --limit 15 --top 4
for f in sorted(glob.glob('outputs/segmentation/seg_*.jpg'))[:3]:
    print(f); display(Image(f))

---
## 10. Empaquetar todo para las slides

Genera un `.zip` con figuras, galería, segmentación y CSVs para descargar y pegar en la presentación.

In [ ]:
!zip -r resultados_slides.zip outputs/figures outputs/gallery outputs/segmentation outputs/*.csv
try:
    from google.colab import files; files.download('resultados_slides.zip')
except Exception as e:
    print('Descarga manual: resultados_slides.zip', e)